# 0 — Preprocess

Flatten QBacMet JSON snapshots, merge with circuit metadata and QFw runtime data, and produce training datasets for MQBac.

**Outputs → `final_data/`**
- `qbacmet_flat.csv` — one row per QBacMet snapshot, all 6 layers flattened
- `best_backend_df.csv` — classification: best backend per config
- `estimate_runtime_df.csv` — regression: runtime for each backend

In [9]:
import os, json, glob, re
from pathlib import Path
import numpy as np
import pandas as pd

# ── Paths (edit if your layout differs) ──
BASE = Path("/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related")
STATS_DIRS = [BASE / "QBacMet" / "stats", BASE / "QBacMet" / "stats_sofar"]
APPS_GLOB  = str(BASE / "applications" / "*.csv")
RUNTIME_CSV  = BASE / "results_analysis" / "qfw_unified_summary.csv"
RUNTIME_JSON = BASE / "results_analysis" / "qfw_unified_data.json"
OUT_DIR = Path("final_data"); OUT_DIR.mkdir(exist_ok=True)

# Skip digit-prefixed test artifacts (0_nq20_..., 28_nq20_...)
SKIP_RE = re.compile(r"^\d+_nq")


## 1. Flatten QBacMet JSONs

In [10]:
def flatten_qbacmet_json(fp):
    """Flatten one QBacMet JSON into a single dict (one CSV row)."""
    with open(fp) as f:
        e = json.load(f)
    row = {"filename": Path(fp).name}

    # ── args ──
    args = e.get("args", {})
    for k, v in args.items():
        row[f"arg_{k}"] = v
    if "optimization_level" not in args:
        row["arg_optimization_level"] = (
            e.get("layers", {}).get("layer_2_transpile", {})
             .get("features", {}).get("optimization_level"))

    row["timestamp"] = e.get("timestamp")
    L = e.get("layers", {})

    # ── Layer 0: SLURM ──
    m0 = L.get("layer_0_slurm", {}).get("metrics", {})
    for k in ["num_nodes", "num_cpus", "num_tasks"]:
        row[f"slurm_{k}"] = m0.get(k)
    f0 = L.get("layer_0_slurm", {}).get("features", {})
    row["slurm_cluster"] = f0.get("frontend_cluster_name")
    row["slurm_account"] = f0.get("account")

    # ── Layer 1: Algorithm ──
    m1 = L.get("layer_1_algorithm", {}).get("metrics", {})
    for k in ["depth", "width", "num_qubits", "num_clbits", "num_ops",
              "num_2q_gates", "num_cliffords", "num_non_cliffords",
              "critical_path_length", "connected_components",
              "num_parameters", "shots"]:
        row[f"algo_{k}"] = m1.get(k)
    gc = m1.get("gate_counts", {})
    if isinstance(gc, dict):
        for g, cnt in gc.items():
            row[f"algo_gate_{g}"] = cnt

    # ── Layer 2: Transpile ──
    f2 = L.get("layer_2_transpile", {}).get("features", {})
    row["transpile_opt_level"] = f2.get("optimization_level")
    m2 = L.get("layer_2_transpile", {}).get("metrics", {})
    for k in ["hw_circ_depth", "hw_circ_width", "hw_circ_num_ops",
              "hw_circ_num_2q_gates", "hw_circ_swaps",
              "hw_circ_connected_components",
              "hw_circ_connectivity_degree_max",
              "hw_circ_connectivity_degree_avg"]:
        row[k] = m2.get(k)
    hgc = m2.get("hw_circ_gate_counts", {})
    if isinstance(hgc, dict):
        for g, cnt in hgc.items():
            row[f"hw_gate_{g}"] = cnt

    # ── Layer 3: Backend ──
    f3 = L.get("layer_3_backend", {}).get("features", {})
    for k in ["backend_name", "backend_type", "backend_is_simulator"]:
        row[f"be_{k}"] = f3.get(k)

    # ── Layer 4: Execution ──
    m4 = L.get("layer_4_execution", {}).get("metrics", {})
    for k in ["openmp_num_threads", "mpi_rank", "mpi_size", "transpile_time"]:
        row[f"exec_{k}"] = m4.get(k)

    # ── Derived ratios ──
    ad = row.get("algo_depth"); hd = row.get("hw_circ_depth")
    if ad and hd:
        try: row["depth_ratio"] = hd / ad
        except (TypeError, ZeroDivisionError): pass
    a2 = row.get("algo_num_2q_gates"); h2 = row.get("hw_circ_num_2q_gates")
    if a2 and h2:
        try: row["twoq_ratio"] = h2 / a2
        except (TypeError, ZeroDivisionError): pass

    return row


# ── Load all JSONs ──
rows = []
skipped = 0
for stats_dir in STATS_DIRS:
    if not stats_dir.exists():
        print(f"[warn] {stats_dir} not found"); continue
    for fp in sorted(stats_dir.glob("*.json")):
        if SKIP_RE.match(fp.name):
            skipped += 1; continue
        try:
            rows.append(flatten_qbacmet_json(fp))
        except Exception as ex:
            print(f"[err] {fp.name}: {ex}")

qbacmet_flat = pd.DataFrame(rows)

# ── Normalize benchmark names: nqmatrix* variants are HHL circuits ──
nq_mask = qbacmet_flat["arg_benchmark_name"].str.startswith("nqmatrix", na=False)
if nq_mask.any():
    print(f"Renaming {nq_mask.sum()} nqmatrix* rows → hhl")
    qbacmet_flat.loc[nq_mask, "arg_benchmark_name"] = "hhl"

print(f"Loaded {len(qbacmet_flat)} QBacMet snapshots ({skipped} test artifacts skipped)")
print(f"Columns: {qbacmet_flat.shape[1]}")

Renaming 19 nqmatrix* rows → hhl
Loaded 1591 QBacMet snapshots (58 test artifacts skipped)
Columns: 76


In [11]:
print("── Benchmarks ──")
print(qbacmet_flat["arg_benchmark_name"].value_counts().to_string())
print("\n── Backends ──")
print(qbacmet_flat["arg_simulator_type"].value_counts().to_string())
print("\n── Sub-backends ──")
print(qbacmet_flat["arg_sub_backend"].value_counts().to_string())
print("\n── Qubit counts ──")
print(sorted(qbacmet_flat["arg_number_of_qubits"].dropna().unique()))
print("\n── Opt levels ──")
print(qbacmet_flat["arg_optimization_level"].value_counts(dropna=False).to_string())
print(f"\n── Shape: {qbacmet_flat.shape} ──")
qbacmet_flat.head(3)


── Benchmarks ──
arg_benchmark_name
ghz            392
tfim           363
ham            208
mermin_bell    181
bit_code       174
phase_code     174
hhl             99

── Backends ──
arg_simulator_type
ionq       1087
ibmq        465
nwqsim       29
qtensor      10

── Sub-backends ──
arg_sub_backend
ibm_miami             291
ideal                 179
aria-1                173
aria-2                172
forte-1               172
forte-enterprise-1    172
harmony               172
ibm_boston            133
simulator              47
ibm_torino             41
MPI                    29
numpy                  10

── Qubit counts ──
[np.int64(4), np.int64(5), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(20), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(39), np.int64(45)]

── Opt levels ──


,filename,arg_benchmark_name,arg_number_of_qubits,arg_simulator_type,arg_sub_backend,arg_device,arg_run_mode,arg_number_of_iterations,arg_optimization_level,timestamp,...,algo_gate_p,algo_gate_barrier,hw_gate_barrier,hw_gate_rx,algo_gate_reset,hw_gate_reset,hw_gate_gpi2,hw_gate_ms,hw_gate_gpi,hw_gate_ry
0,ghz_nq15_ibmq_ibm_boston_CPU_sync_itrs1_opt0_2...,ghz,15,ibmq,ibm_boston,CPU,sync,1,0.0,2026-05-03T00:11:12.755135Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ghz_nq15_ibmq_ibm_boston_CPU_sync_itrs1_opt1_2...,ghz,15,ibmq,ibm_boston,CPU,sync,1,1.0,2026-05-03T00:11:10.499071Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ghz_nq15_ibmq_ibm_boston_CPU_sync_itrs1_opt2_2...,ghz,15,ibmq,ibm_boston,CPU,sync,1,2.0,2026-05-03T00:11:16.783338Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Load Circuit Metadata & QFw Runtime Data

In [12]:
# ── Circuit metadata from bench_meta_gen ──
csv_files = sorted(glob.glob(APPS_GLOB))
print(f"Found {len(csv_files)} circuit metadata CSVs")
if csv_files:
    benchmark_meta_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
    benchmark_meta_df = benchmark_meta_df.drop_duplicates(subset=["benchmark", "n_qubits"])
    print(f"  benchmark_meta_df: {benchmark_meta_df.shape}")
else:
    raise FileNotFoundError(f"No CSVs at {APPS_GLOB}")

# ── QFw aggregated runtime (for classification — pick best backend by mean) ──
if RUNTIME_CSV.exists():
    backend_runtime_df = pd.read_csv(RUNTIME_CSV)
    print(f"  backend_runtime_df (aggregated): {backend_runtime_df.shape}")
else:
    raise FileNotFoundError(f"Runtime CSV not found: {RUNTIME_CSV}")

# ── QFw per-sample runtime (for regression — one row per timing measurement) ──
if RUNTIME_JSON.exists():
    with open(RUNTIME_JSON) as f:
        raw_data = json.load(f)

    persample_rows = []
    def _flatten_json(obj, keys=()):
        if isinstance(obj, list):
            bench, size, backend, sub_backend, device, run_mode, n_nodes, n_procs = keys
            for t_ms in obj:
                persample_rows.append({
                    "benchmark": bench, "size": int(size),
                    "backend": backend, "sub_backend": sub_backend,
                    "device": device, "run_mode": run_mode,
                    "n_nodes": int(n_nodes), "n_processes": int(n_procs),
                    "runtime_ms": t_ms,
                })
        elif isinstance(obj, dict):
            for k, v in obj.items():
                _flatten_json(v, keys + (k,))
    _flatten_json(raw_data)

    backend_runtime_persample_df = pd.DataFrame(persample_rows)
    print(f"  backend_runtime_persample_df (per-sample): {backend_runtime_persample_df.shape}")
    print(f"  Backends: {sorted(backend_runtime_persample_df['backend'].unique().tolist())}")
else:
    raise FileNotFoundError(f"Runtime JSON not found: {RUNTIME_JSON}")

display(benchmark_meta_df.head(3))
display(backend_runtime_persample_df.head(3))


Found 5 circuit metadata CSVs
  benchmark_meta_df: (84, 59)
  backend_runtime_df (aggregated): (638, 11)
  backend_runtime_persample_df (per-sample): (3591, 9)
  Backends: ['ibmq', 'ionq', 'nwqsim', 'qiskitaer', 'qtensor', 'tnqvm']


,benchmark,n_qubits,depth,width,n_clbits,n_ops,n_gates,n_parameters,n_connected_components,n_measure_ops,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,ghz,11,12,11,11,22,22,0,1,11,...,0,1,11,0,0,0,0,0,0,0
1,ghz,12,13,12,12,24,24,0,1,12,...,0,1,12,0,0,0,0,0,0,0
2,ghz,13,14,13,13,26,26,0,1,13,...,0,1,13,0,0,0,0,0,0,0


,benchmark,size,backend,sub_backend,device,run_mode,n_nodes,n_processes,runtime_ms
0,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,109117.286205
1,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,103897.316456
2,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,18630.385160


## 3. Build Classification Dataset

For each `(benchmark, size, n_nodes, n_processes)` choose the backend with the lowest mean runtime.

In [13]:
group_cols = ["benchmark", "size", "n_nodes", "n_processes"]
idx = backend_runtime_df.groupby(group_cols)["mean_ms"].idxmin()
best_rows = backend_runtime_df.loc[
    idx, group_cols + ["backend", "sub_backend", "mean_ms"]
] .copy()
best_rows = best_rows.rename(
    columns={"backend": "best_backend", "mean_ms": "best_runtime"}
)

# Merge with circuit metadata (size <-> n_qubits)
meta_for_join = benchmark_meta_df.copy()
meta_for_join["size"] = meta_for_join["n_qubits"].astype("Int64")
best_backend_df = pd.merge(
    best_rows, meta_for_join, how="left", on=["benchmark", "size"]
)

n_miss = best_backend_df["depth"].isna().sum()
print(f"best_backend_df: {best_backend_df.shape}")
if n_miss:
    print(f"  {n_miss} rows missing circuit metadata (DQAOA/QAOA size mismatch)")
best_backend_df.head(5)


best_backend_df: (233, 65)
  154 rows missing circuit metadata (DQAOA/QAOA size mismatch)


,benchmark,size,n_nodes,n_processes,best_backend,sub_backend,best_runtime,n_qubits,depth,width,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,bit_code,4,1,8,ibmq,ibm_torino,50.197601,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,bit_code,9,1,8,ibmq,ibm_torino,51.785946,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,bit_code,15,1,8,ibmq,ibm_torino,48.943520,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bit_code,16,1,8,ibmq,ibm_torino,53.647757,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,bit_code,20,1,8,ibmq,ibm_torino,56.928635,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Build Regression Dataset

All `(benchmark, size, backend, ...)` rows with individual `runtime_ms` as the target — one row per timing sample.


In [14]:
estimate_runtime_df = pd.merge(
    backend_runtime_persample_df, meta_for_join, how="left", on=["benchmark", "size"]
)
n_miss = estimate_runtime_df["depth"].isna().sum()
print(f"estimate_runtime_df: {estimate_runtime_df.shape}")
if n_miss:
    pct = 100 * n_miss / len(estimate_runtime_df)
    print(f"  {n_miss} rows ({pct:.1f}%) missing circuit metadata")
print(f"  Target column: runtime_ms")
estimate_runtime_df.head(5)


estimate_runtime_df: (3591, 67)
  1778 rows (49.5%) missing circuit metadata
  Target column: runtime_ms


,benchmark,size,backend,sub_backend,device,run_mode,n_nodes,n_processes,runtime_ms,n_qubits,...,gate_cz,gate_h,gate_measure,gate_p,gate_rx,gate_ry,gate_rz,gate_s,gate_swap,gate_u2
0,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,109117.286205,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,103897.316456,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,18630.385160,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,7470.187664,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ghz,4,ibmq,ibm_boston,CPU,sync,1,8,7640.360355,4.0,...,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4b. Merge Cloud History (IBM / IonQ)

If `get_more_data.ipynb` has been run and saved `historic_runs_for_estimate_runtime_df.csv`, append those rows into `estimate_runtime_df` so all downstream modeling sees both HPC sim runs and real cloud runs.

In [15]:
# ── 4b.1: Legacy unified cloud history (from get_more_data.ipynb) ──
import ast

CLOUD_CSV = OUT_DIR / "historic_runs_for_estimate_runtime_df.csv"

JSON_METRIC_KEYS = {
    "depth": ["depth", "algo_depth"],
    "width": ["width", "algo_width"],
    "n_ops": ["n_ops", "num_ops", "algo_num_ops"],
    "single_qubit_gates": ["single_qubit_gates", "num_1q_gates", "algo_num_1q_gates"],
    "two_qubit_gates": ["two_qubit_gates", "num_2q_gates", "algo_num_2q_gates"],
    "three_qubit_gates": ["three_qubit_gates", "num_3q_gates", "algo_num_3q_gates"],
    "measure_ops": ["measure_ops", "num_measure_ops", "measurements"],
    "critical_path_length": ["critical_path_length", "algo_critical_path_length"],
    "connected_components": ["connected_components", "algo_connected_components"],
    "num_parameters": ["num_parameters", "algo_num_parameters"],
    "num_cliffords": ["num_cliffords", "algo_num_cliffords", "clifford_count"],
    "num_non_cliffords": ["num_non_cliffords", "algo_num_non_cliffords", "nonclifford_count"],
}
RAW_PROVENANCE_COLS = {
    "job_id", "status", "created_at", "started_at", "ended_at",
    "benchmark_hint", "session_id",
    "gate_counts_json", "circuit_metrics_json", "hw_circ_gate_counts_json",
    # Infrastructure/provenance metadata that should not drive modeling.
    "timestamp", "filename", "slurm_cluster", "slurm_account",
    "slurm_num_nodes", "slurm_num_cpus", "slurm_num_tasks",
}


def _loads_maybe(value):
    if isinstance(value, dict):
        return value
    if value is None or pd.isna(value):
        return None
    if not isinstance(value, str):
        return None
    text = value.strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return None
    for loader in (json.loads, ast.literal_eval):
        try:
            parsed = loader(text)
        except Exception:
            continue
        if isinstance(parsed, dict):
            return parsed
    return None


def _find_metric(payload, candidates):
    if not isinstance(payload, dict):
        return None
    stack = [payload]
    while stack:
        current = stack.pop()
        if not isinstance(current, dict):
            continue
        for key in candidates:
            if key in current and current[key] is not None:
                return current[key]
        for value in current.values():
            if isinstance(value, dict):
                stack.append(value)
    return None


def _coalesce_numeric(df, column, values):
    numeric_values = pd.to_numeric(values, errors="coerce")
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(numeric_values)
    else:
        df[column] = numeric_values


def _enrich_runtime_frame(df):
    if df.empty:
        return df

    df = df.copy()

    if "gate_counts_json" in df.columns:
        gate_payloads = df["gate_counts_json"].apply(_loads_maybe)
        n_ops_from_counts = gate_payloads.apply(
            lambda payload: sum(float(v) for v in payload.values())
            if isinstance(payload, dict) else np.nan
        )
        _coalesce_numeric(df, "n_ops", n_ops_from_counts)
        measure_from_counts = gate_payloads.apply(
            lambda payload: payload.get("measure", payload.get("measure_ops"))
            if isinstance(payload, dict) else np.nan
        )
        _coalesce_numeric(df, "measure_ops", measure_from_counts)

    if "circuit_metrics_json" in df.columns:
        metric_payloads = df["circuit_metrics_json"].apply(_loads_maybe)
        for target_col, candidates in JSON_METRIC_KEYS.items():
            extracted = metric_payloads.apply(lambda payload: _find_metric(payload, candidates))
            if extracted.notna().any():
                _coalesce_numeric(df, target_col, extracted)

    created_dt = pd.to_datetime(df["created_at"], errors="coerce", utc=True) if "created_at" in df.columns else None
    started_dt = pd.to_datetime(df["started_at"], errors="coerce", utc=True) if "started_at" in df.columns else None
    ended_dt = pd.to_datetime(df["ended_at"], errors="coerce", utc=True) if "ended_at" in df.columns else None

    if created_dt is not None and started_dt is not None:
        _coalesce_numeric(df, "queue_wait_ms", (started_dt - created_dt).dt.total_seconds() * 1000.0)
    if started_dt is not None and ended_dt is not None:
        _coalesce_numeric(df, "execution_window_ms", (ended_dt - started_dt).dt.total_seconds() * 1000.0)
    if created_dt is not None and ended_dt is not None:
        _coalesce_numeric(df, "submit_to_end_ms", (ended_dt - created_dt).dt.total_seconds() * 1000.0)

    return df


def _drop_raw_training_columns(df):
    drop_cols = []
    for col in df.columns:
        if col in RAW_PROVENANCE_COLS:
            drop_cols.append(col)
        elif col.endswith("_json") or col.endswith("_json_qf"):
            drop_cols.append(col)
        elif "qasm_file" in col:
            drop_cols.append(col)
        elif col.startswith("slurm_"):
            drop_cols.append(col)
        elif col in {"timestamp", "filename"}:
            drop_cols.append(col)
        elif col.startswith("arg_slurm_"):
            drop_cols.append(col)
    if drop_cols:
        df = df.drop(columns=sorted(set(drop_cols)), errors="ignore")
    return df


# Rebuild from the base per-sample table each time so this cell is idempotent.
estimate_runtime_df = pd.merge(
    backend_runtime_persample_df,
    meta_for_join,
    how="left",
    on=["benchmark", "size"],
)
print(f"Reset estimate_runtime_df to base per-sample shape: {estimate_runtime_df.shape}")

if CLOUD_CSV.exists():
    cloud_df = pd.read_csv(CLOUD_CSV)
    cloud_df = _enrich_runtime_frame(cloud_df)
    core_cols = ["benchmark", "size", "backend", "sub_backend", "device",
                 "run_mode", "n_nodes", "n_processes", "runtime_ms"]
    extra_cols = [c for c in cloud_df.columns if c not in core_cols]
    cloud_slim = cloud_df[core_cols + extra_cols].copy()
    cloud_slim["runtime_ms"] = pd.to_numeric(cloud_slim["runtime_ms"], errors="coerce")
    cloud_slim = cloud_slim.dropna(subset=["runtime_ms"])
    estimate_runtime_df = pd.concat([estimate_runtime_df, cloud_slim],
                                    ignore_index=True, sort=False)
    print(f"Merged {len(cloud_slim)} unified-cloud rows -> estimate_runtime_df now {estimate_runtime_df.shape}")
else:
    print(f"[info] No unified cloud history at {CLOUD_CSV} - skipping.")

# ── 4b.2: Per-collaborator <USER>_data.csv files (from get_more_data_ibmq.ipynb) ──
RESERVED = {CLOUD_CSV.name,
            "estimate_runtime_df.csv",
            "best_backend_df.csv",
            "qbacmet_flat.csv",
            "historic_runs_unified.csv"}

user_csvs = [p for p in sorted(OUT_DIR.glob("*_data.csv")) if p.name not in RESERVED]
print(f"\nFound {len(user_csvs)} per-user data CSVs in {OUT_DIR}/")

merged_rows = 0
for p in user_csvs:
    try:
        udf = pd.read_csv(p)
    except Exception as e:
        print(f"  [skip] {p.name}: {e}")
        continue
    if udf.empty or "runtime_ms" not in udf.columns:
        print(f"  [skip] {p.name}: no runtime_ms column")
        continue

    udf = _enrich_runtime_frame(udf)
    udf["runtime_ms"] = pd.to_numeric(udf["runtime_ms"], errors="coerce")
    udf = udf.dropna(subset=["runtime_ms"])
    if udf.empty:
        print(f"  [skip] {p.name}: 0 rows with runtime_ms")
        continue

    udf_out = pd.DataFrame({
        "benchmark":    udf.get("benchmark", "ibmq_cloud"),
        "size":         pd.to_numeric(udf.get("n_qubits"), errors="coerce"),
        "backend":      udf.get("provider", "ibmq"),
        "sub_backend":  udf.get("backend"),
        "device":       udf.get("device", "QPU"),
        "run_mode":     udf.get("run_mode", "sync"),
        "n_nodes":      pd.to_numeric(udf["n_nodes"] if "n_nodes" in udf.columns else pd.Series(1, index=udf.index), errors="coerce").fillna(1).astype(int),
        "n_processes":  pd.to_numeric(udf["n_processes"] if "n_processes" in udf.columns else pd.Series(1, index=udf.index), errors="coerce").fillna(1).astype(int),
        "runtime_ms":   udf["runtime_ms"],
    })

    for col in [
        "depth", "width", "n_ops", "single_qubit_gates", "two_qubit_gates",
        "three_qubit_gates", "measure_ops", "critical_path_length",
        "connected_components", "num_cliffords", "num_non_cliffords",
        "num_parameters", "queue_wait_ms", "execution_window_ms",
        "submit_to_end_ms",
    ]:
        if col in udf.columns:
            udf_out[col] = udf[col]

    udf_out = udf_out.dropna(subset=["size"])
    udf_out["size"] = udf_out["size"].astype(int)

    estimate_runtime_df = pd.concat([estimate_runtime_df, udf_out],
                                    ignore_index=True, sort=False)
    merged_rows += len(udf_out)
    print(f"  [ok]  {p.name}: +{len(udf_out)} rows (sub_backends: {sorted(udf_out['sub_backend'].dropna().unique().tolist())})")

print(f"\nTotal per-user rows merged: {merged_rows}")
print(f"estimate_runtime_df now: {estimate_runtime_df.shape}")
print(f"Backends after all merges: {sorted(estimate_runtime_df['backend'].dropna().unique().tolist())}")

# ── 4b.3: Merge consolidated raw_data summary from results_analysis/ ──
QFW_SUMMARY_CSV = Path("/lustre/orion/gen008/proj-shared/qhpc/srikar/qfw_related/results_analysis/qfw_unified_summary.csv")
if QFW_SUMMARY_CSV.exists():
    try:
        qfw = pd.read_csv(QFW_SUMMARY_CSV)
        needed = ["benchmark", "size", "backend", "sub_backend", "device", "run_mode",
                  "n_nodes", "n_processes", "mean_ms"]
        missing = [c for c in needed if c not in qfw.columns]
        if missing:
            print(f"[warn] {QFW_SUMMARY_CSV.name} missing columns {missing}; skipping merge.")
        else:
            qfw = qfw.copy()
            qfw["runtime_ms"] = pd.to_numeric(qfw["mean_ms"], errors="coerce")
            qfw = qfw.dropna(subset=["runtime_ms", "size"])

            for c in ["benchmark", "backend", "sub_backend", "device", "run_mode"]:
                qfw[c] = qfw[c].astype(str).str.strip()
            for c in ["benchmark", "backend", "sub_backend", "device", "run_mode"]:
                estimate_runtime_df[c] = estimate_runtime_df[c].astype(str).str.strip()

            qfw["size"] = pd.to_numeric(qfw["size"], errors="coerce")
            qfw["n_nodes"] = pd.to_numeric(qfw["n_nodes"], errors="coerce").fillna(1).astype(int)
            qfw["n_processes"] = pd.to_numeric(qfw["n_processes"], errors="coerce").fillna(1).astype(int)

            key_cols = ["benchmark", "size", "backend", "sub_backend", "device", "run_mode", "n_nodes", "n_processes"]
            _base_keys = estimate_runtime_df[key_cols].copy()
            _base_keys["size"] = pd.to_numeric(_base_keys["size"], errors="coerce")
            _base_keys["n_nodes"] = pd.to_numeric(_base_keys["n_nodes"], errors="coerce").fillna(1).astype(int)
            _base_keys["n_processes"] = pd.to_numeric(_base_keys["n_processes"], errors="coerce").fillna(1).astype(int)
            existing_keys = set(map(tuple, _base_keys.dropna(subset=["size"]).itertuples(index=False, name=None)))

            qfw = qfw.dropna(subset=["size"])
            qfw_keys = list(map(tuple, qfw[key_cols].itertuples(index=False, name=None)))
            keep_mask = [k not in existing_keys for k in qfw_keys]
            qfw_new = qfw.loc[keep_mask, :].copy()

            if not qfw_new.empty:
                keep_cols = key_cols + ["runtime_ms"]
                for extra_col in ["samples", "std_ms"]:
                    if extra_col in qfw_new.columns:
                        keep_cols.append(extra_col)
                estimate_runtime_df = pd.concat([estimate_runtime_df, qfw_new[keep_cols]], ignore_index=True, sort=False)
            print(f"Merged {len(qfw_new)} new rows from {QFW_SUMMARY_CSV.name} -> estimate_runtime_df now {estimate_runtime_df.shape}")
            if "benchmark" in qfw_new.columns and "backend" in qfw_new.columns and "sub_backend" in qfw_new.columns:
                _chk = qfw_new[(qfw_new["benchmark"].astype(str).str.lower() == "ghz") &
                               (qfw_new["backend"].astype(str).str.lower() == "ibmq")]
                if not _chk.empty:
                    print("Added GHZ IBMQ sub_backends from unified summary:",
                          sorted(_chk["sub_backend"].dropna().astype(str).unique().tolist()))
    except Exception as e:
        print(f"[warn] Failed to merge {QFW_SUMMARY_CSV}: {e}")
else:
    print(f"[info] {QFW_SUMMARY_CSV} not found - skipping unified raw_data merge.")

print(f"Backends after unified-summary merge: {sorted(estimate_runtime_df['backend'].dropna().unique().tolist())}")

# ── 4c: Enrich runtime rows with QBacMet features on shared keys ──
# First merge backend-specific QBacMet rows, then add benchmark/size-level
# circuit features so cloud and HPC rows share a broader common feature space.
qf_join = qbacmet_flat.rename(columns={
    "arg_benchmark_name": "benchmark",
    "arg_number_of_qubits": "size",
    "arg_simulator_type": "backend",
    "arg_sub_backend": "sub_backend",
}).copy()

for col in ["benchmark", "backend", "sub_backend"]:
    if col in qf_join.columns:
        qf_join[col] = qf_join[col].astype(str).str.strip()
for col in ["benchmark", "backend", "sub_backend"]:
    if col in estimate_runtime_df.columns:
        estimate_runtime_df[col] = estimate_runtime_df[col].astype(str).str.strip()

qf_join["size"] = pd.to_numeric(qf_join["size"], errors="coerce")
estimate_runtime_df["size"] = pd.to_numeric(estimate_runtime_df["size"], errors="coerce")

qf_key_cols = ["benchmark", "size", "backend", "sub_backend"]
qf_num_cols = [c for c in qf_join.select_dtypes(include=[np.number]).columns if c not in qf_key_cols]
qf_other_cols = [
    c for c in qf_join.columns
    if c not in qf_key_cols + qf_num_cols
    and c not in RAW_PROVENANCE_COLS
    and not c.endswith("_json")
    and "qasm_file" not in c
    and not c.startswith("slurm_")
    and c not in {"timestamp", "filename"}
]

agg_spec = {c: "mean" for c in qf_num_cols}
for c in qf_other_cols:
    agg_spec[c] = "first"

qf_agg = qf_join.groupby(qf_key_cols, dropna=False).agg(agg_spec).reset_index()

feature_cols = [c for c in qf_agg.columns if c not in qf_key_cols]
rename_map = {c: (f"{c}_qf" if c in estimate_runtime_df.columns else c) for c in feature_cols}
qf_agg = qf_agg.rename(columns=rename_map)

before_cols = set(estimate_runtime_df.columns)
estimate_runtime_df = pd.merge(
    estimate_runtime_df,
    qf_agg,
    how="left",
    on=qf_key_cols,
    validate="m:1",
)
added_cols = [c for c in estimate_runtime_df.columns if c not in before_cols]

# Circuit-level QBacMet metrics that remain meaningful even without an exact
# backend/sub-backend match. These are the main modeling inputs.
circuit_metric_map = {
    "algo_depth": "depth",
    "algo_width": "width",
    "algo_num_ops": "n_ops",
    "algo_num_2q_gates": "two_qubit_gates",
    "algo_num_cliffords": "clifford_count",
    "algo_num_non_cliffords": "nonclifford_count",
    "algo_critical_path_length": "critical_path_length",
    "algo_connected_components": "connected_components",
    "algo_num_parameters": "num_parameters",
}

qf_circuit_cols = ["benchmark", "size"] + [c for c in circuit_metric_map if c in qf_join.columns]
qf_circuit_agg = (
    qf_join[qf_circuit_cols]
    .groupby(["benchmark", "size"], dropna=False)
    .mean(numeric_only=True)
    .reset_index()
    .rename(columns={src: f"{dst}_qf" for src, dst in circuit_metric_map.items() if src in qf_circuit_cols})
)

estimate_runtime_df = pd.merge(
    estimate_runtime_df,
    qf_circuit_agg,
    how="left",
    on=["benchmark", "size"],
    suffixes=("", "_pair"),
)

# Build canonical circuit metrics by preferring explicit row values first, then
# QBacMet pair-level backfills, then backend-specific QBacMet enrichments.
canonical_metric_sources = {
    "depth": ["depth", "depth_qf", "algo_depth", "algo_depth_qf"],
    "width": ["width", "width_qf", "algo_width", "algo_width_qf"],
    "n_ops": ["n_ops", "n_ops_qf", "algo_num_ops", "algo_num_ops_qf"],
    "two_qubit_gates": ["two_qubit_gates", "two_qubit_gates_qf", "algo_num_2q_gates", "algo_num_2q_gates_qf"],
    "clifford_count": ["clifford_count", "num_cliffords", "clifford_count_qf", "algo_num_cliffords", "algo_num_cliffords_qf"],
    "nonclifford_count": ["nonclifford_count", "num_non_cliffords", "nonclifford_count_qf", "algo_num_non_cliffords", "algo_num_non_cliffords_qf"],
    "critical_path_length": ["critical_path_length", "critical_path_length_qf", "algo_critical_path_length", "algo_critical_path_length_qf"],
    "connected_components": ["connected_components", "connected_components_qf", "algo_connected_components", "algo_connected_components_qf"],
    "num_parameters": ["num_parameters", "num_parameters_qf", "algo_num_parameters", "algo_num_parameters_qf"],
}

for canonical_col, source_cols in canonical_metric_sources.items():
    present = [c for c in source_cols if c in estimate_runtime_df.columns]
    if not present:
        continue
    canonical = pd.Series(np.nan, index=estimate_runtime_df.index, dtype="float64")
    for src in present:
        canonical = canonical.fillna(pd.to_numeric(estimate_runtime_df[src], errors="coerce"))
    estimate_runtime_df[canonical_col] = canonical

coverage_col = "depth" if "depth" in estimate_runtime_df.columns else None
if coverage_col is not None:
    cov = 100.0 * estimate_runtime_df[coverage_col].notna().mean()
    print(f"Canonical circuit-feature coverage: {cov:.1f}% rows (using {coverage_col})")
print(f"Added {len(added_cols)} backend-specific QBacMet feature columns to estimate_runtime_df")
print(f"Added {len([c for c in canonical_metric_sources if c in estimate_runtime_df.columns])} canonical circuit metrics")

# ── 4d: Rebuild classification targets from combined runtime pool ──
# Backend selection is defined at the HPC workflow level, so group by
# (benchmark, size, n_nodes, n_processes) before picking the fastest backend.
runtime_target = "runtime_ms" if "runtime_ms" in estimate_runtime_df.columns else "mean_ms"
base_cols = ["benchmark", "size", "n_nodes", "n_processes", "backend", "sub_backend"]
base_cols = [c for c in base_cols if c in estimate_runtime_df.columns]

backend_runtime_aug_df = (
    estimate_runtime_df
    .dropna(subset=[runtime_target])
    .groupby(base_cols, dropna=False)[runtime_target]
    .mean()
    .reset_index()
    .rename(columns={runtime_target: "mean_ms"})
)

group_cols_aug = [c for c in ["benchmark", "size", "n_nodes", "n_processes"] if c in backend_runtime_aug_df.columns]
idx_aug = backend_runtime_aug_df.groupby(group_cols_aug)["mean_ms"].idxmin()
best_rows_aug = backend_runtime_aug_df.loc[idx_aug, group_cols_aug + ["backend", "sub_backend", "mean_ms"]].copy()
best_rows_aug = best_rows_aug.rename(columns={"backend": "best_backend", "mean_ms": "best_runtime"})

best_backend_df = pd.merge(best_rows_aug, meta_for_join, how="left", on=["benchmark", "size"])
n_miss_best = best_backend_df["depth"].isna().sum()
print(f"\nRebuilt best_backend_df from combined runtimes: {best_backend_df.shape}")
if n_miss_best:
    print(f"  {n_miss_best} rows missing circuit metadata after rebuild")

# Mirror the same canonical metric backfills into the classification table.
best_tmp = best_backend_df.rename(columns={"best_backend": "backend"})
best_tmp = pd.merge(best_tmp, qf_agg, how="left", on=qf_key_cols, validate="m:1")
best_tmp = pd.merge(best_tmp, qf_circuit_agg, how="left", on=["benchmark", "size"], suffixes=("", "_pair"))
for canonical_col, source_cols in canonical_metric_sources.items():
    present = [c for c in source_cols if c in best_tmp.columns]
    if not present:
        continue
    canonical = pd.Series(np.nan, index=best_tmp.index, dtype="float64")
    for src in present:
        canonical = canonical.fillna(pd.to_numeric(best_tmp[src], errors="coerce"))
    best_tmp[canonical_col] = canonical
best_backend_df = best_tmp.rename(columns={"backend": "best_backend"})
coverage_col_best = "depth" if "depth" in best_backend_df.columns else None
if coverage_col_best is not None:
    cov_best = 100.0 * best_backend_df[coverage_col_best].notna().mean()
    print(f"Canonical circuit-feature coverage in best_backend_df: {cov_best:.1f}% rows (using {coverage_col_best})")

raw_regression_cols = set(estimate_runtime_df.columns)
raw_classification_cols = set(best_backend_df.columns)
estimate_runtime_df = _drop_raw_training_columns(estimate_runtime_df)
best_backend_df = _drop_raw_training_columns(best_backend_df)
print(f"Dropped {len(raw_regression_cols - set(estimate_runtime_df.columns))} raw/provenance columns from estimate_runtime_df")
print(f"Dropped {len(raw_classification_cols - set(best_backend_df.columns))} raw/provenance columns from best_backend_df")

Reset estimate_runtime_df to base per-sample shape: (3591, 67)
Merged 80 unified-cloud rows -> estimate_runtime_df now (3671, 75)

Found 1 per-user data CSVs in final_data/
  [ok]  schundu3_data.csv: +200 rows (sub_backends: ['ibm_boston', 'ibm_fez', 'ibm_kingston', 'ibm_miami', 'ibm_pittsburgh', 'ibm_torino'])

Total per-user rows merged: 200
estimate_runtime_df now: (3871, 81)
Backends after all merges: ['ibmq', 'ionq', 'nwqsim', 'qiskitaer', 'qtensor', 'tnqvm']
Merged 0 new rows from qfw_unified_summary.csv -> estimate_runtime_df now (3871, 81)
Backends after unified-summary merge: ['ibmq', 'ionq', 'nwqsim', 'qiskitaer', 'qtensor', 'tnqvm']
Canonical circuit-feature coverage: 79.3% rows (using depth)
Added 67 backend-specific QBacMet feature columns to estimate_runtime_df
Added 9 canonical circuit metrics

Rebuilt best_backend_df from combined runtimes: (236, 65)
  157 rows missing circuit metadata after rebuild
Canonical circuit-feature coverage in best_backend_df: 80.9% rows (usin

## 5. Save & Summary

In [16]:
# Save CSVs
qbacmet_flat.to_csv(OUT_DIR / "qbacmet_flat.csv", index=False)
best_backend_df.to_csv(OUT_DIR / "best_backend_df.csv", index=False)
estimate_runtime_df.to_csv(OUT_DIR / "estimate_runtime_df.csv", index=False)

# Final training tables used downstream by 1_analyze.ipynb and 2_model.ipynb
best_backend_df.to_csv(OUT_DIR / "use_this_for_classification.csv", index=False)
estimate_runtime_df.to_csv(OUT_DIR / "use_this_for_regression.csv", index=False)

print("Saved to final_data/:")
for f in sorted(OUT_DIR.glob("*.csv")):
    n = len(pd.read_csv(f, low_memory=False))
    print(f"  {f.name}: {n} rows")

print("\nFeature-label convention used downstream in 2_model.ipynb:")
print("  categorical: <feature> = <category>")
print("  numeric: <feature>")

print(f"\n── Numbers for the paper ──")
print(f"QBacMet snapshots: {len(qbacmet_flat)}")
print(f"Unique benchmarks: {qbacmet_flat['arg_benchmark_name'].nunique()}")
print(f"Unique sim types:  {qbacmet_flat['arg_simulator_type'].nunique()}")
print(f"Unique sub-backends: {qbacmet_flat['arg_sub_backend'].nunique()}")
print(f"Qubit range: {int(qbacmet_flat['arg_number_of_qubits'].min())}"
      f"–{int(qbacmet_flat['arg_number_of_qubits'].max())}")
print(f"Classification rows: {len(best_backend_df)}")
print(f"Regression rows:     {len(estimate_runtime_df)} ({len(backend_runtime_persample_df)} individual timing samples)")

print("\nDownstream inputs:")
print("  use_this_for_classification.csv")
print("  use_this_for_regression.csv")

Saved to final_data/:
  4kl_data_000000_000200.csv: 87 rows
  best_backend_df.csv: 236 rows
  estimate_runtime_df.csv: 3871 rows
  historic_runs_for_estimate_runtime_df.csv: 80 rows
  historic_runs_unified.csv: 80 rows
  qbacmet_flat.csv: 1591 rows
  schundu3_data.csv: 200 rows
  schundu3_data_000000_000200.csv: 200 rows
  skim_data_000000_000200.csv: 200 rows
  use_this_for_classification.csv: 236 rows
  use_this_for_regression.csv: 3871 rows

Feature-label convention used downstream in 2_model.ipynb:
  categorical: <feature> = <category>
  numeric: <feature>

── Numbers for the paper ──
QBacMet snapshots: 1591
Unique benchmarks: 7
Unique sim types:  4
Unique sub-backends: 12
Qubit range: 4–45
Classification rows: 236
Regression rows:     3871 (3591 individual timing samples)

Downstream inputs:
  use_this_for_classification.csv
  use_this_for_regression.csv
